# Quantitative Equity Screener

**A two stage fundamental screen for Indian equities (NSE / BSE)**

## Executive Summary

This notebook implements a systematic screen that narrows the ~7,800 listings on
the Indian exchanges down to a short list of companies which are simultaneously
**high quality**, **high growth**, and **reasonably valued**.

The screen is deliberately split into two stages, for reasons of both cost and
capability:

| | Stage 1: Native Screening | Stage 2: Analytical Refinement |
|---|---|---|
| **Engine** | TradingView scanner API | pandas + yfinance |
| **Work done** | Server side filtering on absolute fundamental thresholds | Relative valuation, historical valuation, derived growth metrics |
| **Why here** | One request filters the entire market, with no bulk download needed | Requires cross sectional context and time series that the scanner cannot express |

Stage 1 answers *"is this a good business growing quickly?"* Stage 2 answers the
harder question, *"and is it currently cheap, both against its peers and against
its own trading history?"*

```
              ~7,800 NSE / BSE listings
                        |
   STAGE 1   Market cap, ROE, ROCE, sales growth (1Y / 5Y), PEG
                        |
              collapse NSE / BSE dual listings
                        |
   STAGE 2a  Relative valuation:   PE < industry benchmark PE
                        |
   STAGE 2b  Historical valuation: PE < own long run average PE
             Derived metrics:      3Y CAGRs, average ROE
                        |
                   Final shortlist
```

**Output:** a ranked DataFrame of surviving candidates with full supporting
metrics, exported to `results.csv`.

**Scope:** this is a candidate generation tool built on reported fundamentals. It
does not assess balance sheet leverage, cash flow quality, or governance, and is
not investment advice.

## 1. Configuration & Dependencies

All tunable parameters are centralised in a single `CONFIG` dictionary so the
screen can be tightened or relaxed without touching pipeline logic. Any
threshold set to `None` disables that filter entirely.

In [1]:
from __future__ import annotations

import logging
import math
import time
from typing import Any

import pandas as pd
import yfinance as yf
from tradingview_screener import Column, Query

logging.getLogger("yfinance").setLevel(logging.CRITICAL)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [2]:
CONFIG: dict[str, Any] = {
    "MARKET": "india",
    "EXCHANGES": ["NSE", "BSE"],
    "PREFERRED_EXCHANGE": "NSE",
    "MAX_ROWS": 2000,

    "MIN_MARKET_CAP": 40_000_000_000,
    "MIN_ROE": 20.0,
    "MIN_ROCE": 20.0,
    "MIN_REVENUE_GROWTH_TTM": 20.0,
    "MIN_REVENUE_CAGR_5Y": 16.0,
    "MIN_NET_INCOME_GROWTH_TTM": None,
    "MIN_NET_INCOME_CAGR_5Y": None,
    "MAX_PEG": 1.3,
    "ROCE_FIELD": "roic",

    "APPLY_INDUSTRY_PE_FILTER": True,
    "INDUSTRY_PE_BENCHMARK": "universe",
    "INDUSTRY_PE_STATISTIC": "median",
    "MIN_PEERS_FOR_INDUSTRY_AVG": 3,
    "KEEP_WHEN_INSUFFICIENT_PEERS": True,
    "PE_OUTLIER_BOUNDS": (0.0, 300.0),

    "APPLY_HISTORICAL_PE_FILTER": True,
    "HISTORY_YEARS": 5,
    "MIN_PE_HISTORY_MONTHS": 24,
    "KEEP_WHEN_NO_PE_HISTORY": False,
    "REPORTING_LAG_DAYS": 60,

    "STAGE2_METRICS": {
        "MIN_NET_INCOME_CAGR_3Y": 20.0,
        "MIN_REVENUE_CAGR_3Y": 17.0,
        "MIN_ROE_5Y_AVG": 20.0,
    },
    "MIN_FY_FOR_AVERAGES": 3,
    "KEEP_WHEN_STAGE2_METRICS_MISSING": True,

    "REQUEST_DELAY_SEC": 0.4,
    "MAX_RETRIES": 2,
    "OUTPUT_CSV": "results.csv",
}

### Field mapping

The scanner exposes several thousand fields with overlapping names and very
different population rates. Two mapping decisions are worth recording:

- **Fiscal year (`_fy`) variants are used for ROE and ROCE.** On the Indian
  large cap universe the plain and `_fq` variants are populated for only ~14% of
  companies, against ~98% for `_fy`. Filtering on the sparser field silently
  discards the majority of valid candidates.
- **The scanner returns `NULL` rather than an error for unrecognised field
  names.** A misspelled field is therefore indistinguishable from a universally
  unreported metric, and will empty the screen without warning. The mapping
  below was validated against the exchange metadata endpoint and against live
  population rates.

In [3]:
FIELDS: dict[str, str] = {
    "name": "name",
    "exchange": "exchange",
    "sector": "sector",
    "industry": "industry",
    "close": "close",
    "market_cap": "market_cap_basic",
    "roe": "return_on_equity_fy",
    "roic": "return_on_invested_capital_fy",
    "roce": "return_on_capital_employed_fy",
    "net_income_growth_ttm": "net_income_yoy_growth_ttm",
    "net_income_cagr_5y": "net_income_cagr_5y",
    "revenue_growth_ttm": "total_revenue_yoy_growth_ttm",
    "revenue_cagr_5y": "total_revenue_cagr_5y",
    "peg": "price_earnings_growth_ttm",
    "pe": "price_earnings_ttm",
}

LABELS: dict[str, str] = {
    "market_cap": "Market Cap",
    "roe": "ROE",
    "roic": "ROIC",
    "roce": "ROCE",
    "net_income_growth_ttm": "Net Profit Growth TTM",
    "net_income_cagr_5y": "Net Profit CAGR 5Y",
    "revenue_growth_ttm": "Sales Growth 1Y",
    "revenue_cagr_5y": "Sales Growth 5Y",
    "peg": "PEG",
}

In [4]:
def crores(value: float) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "n/a"
    return f"{value / 1_00_00_000:,.0f} Cr"


def cagr(latest: float, oldest: float, years: float) -> float | None:
    if years <= 0 or oldest is None or latest is None:
        return None
    if oldest <= 0 or latest <= 0:
        return None
    return ((latest / oldest) ** (1.0 / years) - 1.0) * 100.0


def roce_field() -> str:
    return "roce" if CONFIG["ROCE_FIELD"] == "roce" else "roic"

## 2. Stage 1: Native TradingView Screening (Growth & Quality Filters)

Stage 1 pushes every absolute threshold to the scanner so that filtering happens
server side across the full market in a single request.

Two structural issues are handled here:

1. **Dual listings.** Nearly every Indian company is listed on both the NSE and
   the BSE, so the raw response contains each business twice. Left uncorrected
   this double counts the universe and biases every downstream industry
   aggregate. `deduplicate_listings()` collapses them, preferring the NSE line.
2. **Filter attribution.** When a multi condition screen returns nothing, the
   result alone does not reveal which threshold was binding.
   `run_funnel_analysis()` applies the filters cumulatively and reports the
   survivor count at each step, making the screen diagnosable rather than opaque.

In [5]:
def build_filters() -> list:
    filters = []

    if CONFIG["EXCHANGES"]:
        filters.append(Column(FIELDS["exchange"]).isin(CONFIG["EXCHANGES"]))

    thresholds = [
        ("MIN_MARKET_CAP", "market_cap", "gt"),
        ("MIN_ROE", "roe", "gt"),
        ("MIN_ROCE", roce_field(), "gt"),
        ("MIN_NET_INCOME_GROWTH_TTM", "net_income_growth_ttm", "gt"),
        ("MIN_NET_INCOME_CAGR_5Y", "net_income_cagr_5y", "gt"),
        ("MIN_REVENUE_GROWTH_TTM", "revenue_growth_ttm", "gt"),
        ("MIN_REVENUE_CAGR_5Y", "revenue_cagr_5y", "gt"),
        ("MAX_PEG", "peg", "lt"),
    ]

    for key, field, op in thresholds:
        limit = CONFIG[key]
        if limit is None:
            continue
        column = Column(FIELDS[field])
        filters.append(column > limit if op == "gt" else column < limit)

    return filters


def deduplicate_listings(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "name" not in df.columns:
        return df

    df = df.copy()
    df["_rank"] = (df.get("exchange") != CONFIG["PREFERRED_EXCHANGE"]).astype(int)
    return (
        df.sort_values(["name", "_rank"])
        .drop_duplicates(subset="name", keep="first")
        .drop(columns="_rank")
        .reset_index(drop=True)
    )

In [6]:
def _match_count(filters: list) -> int:
    count, _ = (
        Query()
        .select(FIELDS["name"])
        .set_markets(CONFIG["MARKET"])
        .where(*filters)
        .limit(1)
        .get_scanner_data()
    )
    return count


def run_funnel_analysis() -> pd.DataFrame:
    steps = [
        ("MIN_MARKET_CAP", "market_cap", ">", crores),
        ("MIN_ROE", "roe", ">", None),
        ("MIN_ROCE", roce_field(), ">", None),
        ("MIN_NET_INCOME_GROWTH_TTM", "net_income_growth_ttm", ">", None),
        ("MIN_NET_INCOME_CAGR_5Y", "net_income_cagr_5y", ">", None),
        ("MIN_REVENUE_GROWTH_TTM", "revenue_growth_ttm", ">", None),
        ("MIN_REVENUE_CAGR_5Y", "revenue_cagr_5y", ">", None),
        ("MAX_PEG", "peg", "<", None),
    ]

    applied = [Column(FIELDS["exchange"]).isin(CONFIG["EXCHANGES"])]
    remaining = _match_count(applied)
    rows = [{"Filter": "Exchange (NSE / BSE)", "Threshold": "", "Listings": remaining, "Removed": 0}]

    for key, field, op, formatter in steps:
        limit = CONFIG[key]
        if limit is None:
            continue

        column = Column(FIELDS[field])
        applied.append(column > limit if op == ">" else column < limit)
        survivors = _match_count(applied)

        rows.append(
            {
                "Filter": LABELS.get(field, field),
                "Threshold": f"{op} {formatter(limit) if formatter else f'{limit:g}'}",
                "Listings": survivors,
                "Removed": remaining - survivors,
            }
        )
        remaining = survivors

    return pd.DataFrame(rows)

In [7]:
def fetch_tradingview_data() -> pd.DataFrame:
    fields = list(dict.fromkeys(FIELDS.values()))

    _, df = (
        Query()
        .select(*fields)
        .set_markets(CONFIG["MARKET"])
        .where(*build_filters())
        .limit(CONFIG["MAX_ROWS"])
        .get_scanner_data()
    )

    if df.empty:
        return df

    df = df.rename(columns={v: k for k, v in FIELDS.items()})
    if "ticker" in df.columns:
        df = df.rename(columns={"ticker": "tv_ticker"})

    return deduplicate_listings(df)


def classify_growth_status(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    df["growth_status"] = "High Growth"
    return df

## 3. Stage 2: Industry Relative Valuation & Historical PE Analysis

Stage 1 establishes that a business is good. Stage 2 establishes whether it is
currently available at a sensible price, using two independent tests.

### 3a. Relative valuation

A stock must trade below the benchmark PE of its industry.

The benchmark is drawn from the **full market** above the market cap floor
rather than from the Stage 1 survivors. Benchmarking against survivors is
degenerate: the growth filters are strict enough that most industries retain a
single company, and a lone stock can never sit below its own average, so valid
candidates are eliminated by an artefact of sample size. Setting
`INDUSTRY_PE_BENCHMARK` to `"stage1"` restores the survivor only behaviour.

The **median** is used by default, as PE distributions carry extreme outliers
that distort a mean.

### 3b. Historical valuation

A stock must also trade below its own long run average PE.

The historical series is reconstructed by pairing each month's close with the
most recently *reported* fiscal year EPS, offset by `REPORTING_LAG_DAYS` so that
no observation uses earnings that were not yet public. Dividing historical
prices by present day EPS would be considerably simpler and entirely
meaningless.

Because the provider exposes roughly four annual EPS observations, the
reconstructable window is typically three to four years rather than a full five.
The realised depth is reported per stock in `pe_history_months` and results
below `MIN_PE_HISTORY_MONTHS` are flagged rather than silently judged.

### 3c. Derived growth metrics

Three year CAGRs and multi year average ROE are not exposed by the scanner,
which offers only five year horizons. They are computed here from annual
financial statements.

In [8]:
def fetch_industry_benchmark() -> pd.DataFrame:
    filters = [Column(FIELDS["exchange"]).isin(CONFIG["EXCHANGES"])]
    if CONFIG["MIN_MARKET_CAP"] is not None:
        filters.append(Column(FIELDS["market_cap"]) > CONFIG["MIN_MARKET_CAP"])

    _, df = (
        Query()
        .select(FIELDS["name"], FIELDS["exchange"], FIELDS["industry"], FIELDS["pe"])
        .set_markets(CONFIG["MARKET"])
        .where(*filters)
        .limit(CONFIG["MAX_ROWS"])
        .get_scanner_data()
    )

    df = df.rename(columns={v: k for k, v in FIELDS.items()})
    return deduplicate_listings(df)


def compute_industry_stats(df: pd.DataFrame) -> pd.DataFrame:
    low, high = CONFIG["PE_OUTLIER_BOUNDS"]

    work = df[["industry", "pe"]].copy()
    work["pe"] = pd.to_numeric(work["pe"], errors="coerce")
    work = work[(work["pe"] > low) & (work["pe"] < high)]

    grouped = work.groupby("industry")["pe"]
    statistic = grouped.median() if CONFIG["INDUSTRY_PE_STATISTIC"] == "median" else grouped.mean()

    return pd.DataFrame(
        {"industry_pe": statistic, "industry_peers": grouped.size()}
    ).reset_index()

In [9]:
def apply_industry_pe_filters(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    df = classify_growth_status(df)

    if not CONFIG["APPLY_INDUSTRY_PE_FILTER"]:
        df = df.copy()
        df["industry_pe"] = pd.NA
        df["industry_peers"] = pd.NA
        df["industry_pe_status"] = "skipped"
        return df

    source = fetch_industry_benchmark() if CONFIG["INDUSTRY_PE_BENCHMARK"] == "universe" else df
    if source.empty:
        source = df

    stats = compute_industry_stats(source)
    df = df.drop(columns=[c for c in ("industry_pe", "industry_peers") if c in df.columns])
    df = df.merge(stats, on="industry", how="left")
    df["pe"] = pd.to_numeric(df["pe"], errors="coerce")

    minimum_peers = CONFIG["MIN_PEERS_FOR_INDUSTRY_AVG"]

    def verdict(row: pd.Series) -> str:
        if pd.isna(row["pe"]):
            return "no PE data"
        if pd.isna(row["industry_pe"]) or pd.isna(row["industry_peers"]):
            return "no industry benchmark"
        if row["industry_peers"] < minimum_peers:
            return "insufficient peers"
        return "below industry PE" if row["pe"] < row["industry_pe"] else "above industry PE"

    df["industry_pe_status"] = df.apply(verdict, axis=1)

    accepted = {"below industry PE"}
    if CONFIG["KEEP_WHEN_INSUFFICIENT_PEERS"]:
        accepted |= {"insufficient peers", "no industry benchmark"}

    return df[df["industry_pe_status"].isin(accepted)].reset_index(drop=True)

In [10]:
def yahoo_symbol_candidates(row: pd.Series) -> list[str]:
    base = str(row["name"]).strip()
    primary = ".BO" if row.get("exchange") == "BSE" else ".NS"
    secondary = ".NS" if primary == ".BO" else ".BO"

    spellings = [base]
    if "_" in base:
        spellings += [base.replace("_", "-"), base.replace("_", "&"), base.replace("_", "")]

    candidates = [f"{s}{primary}" for s in spellings] + [f"{s}{secondary}" for s in spellings]
    return list(dict.fromkeys(candidates))


def to_yahoo_symbol(row: pd.Series) -> str:
    return yahoo_symbol_candidates(row)[0]


def _fetch_symbol(symbol: str) -> dict[str, Any] | None:
    ticker = yf.Ticker(symbol)
    prices = ticker.history(
        period=f"{CONFIG['HISTORY_YEARS']}y", interval="1mo", auto_adjust=True
    )
    income = ticker.income_stmt
    balance = ticker.balance_sheet

    if (prices is None or prices.empty) and (income is None or income.empty):
        return None

    return {"prices": prices, "income": income, "balance": balance, "symbol": symbol}


def download_fundamentals(row: pd.Series) -> dict[str, Any] | None:
    for symbol in yahoo_symbol_candidates(row):
        for attempt in range(CONFIG["MAX_RETRIES"] + 1):
            try:
                bundle = _fetch_symbol(symbol)
                if bundle is not None:
                    return bundle
                break
            except Exception:
                if attempt < CONFIG["MAX_RETRIES"]:
                    time.sleep(1.0 + attempt)
    return None

In [11]:
def build_eps_timeline(income: pd.DataFrame) -> pd.DataFrame | None:
    if income is None or income.empty:
        return None

    eps = None
    for key in ("Diluted EPS", "Basic EPS"):
        if key in income.index:
            candidate = pd.to_numeric(income.loc[key], errors="coerce").dropna()
            if not candidate.empty:
                eps = candidate
                break

    if eps is None or eps.empty:
        return None

    effective = pd.to_datetime(eps.index).astype("datetime64[ns]") + pd.Timedelta(
        days=CONFIG["REPORTING_LAG_DAYS"]
    )
    timeline = pd.DataFrame({"effective": effective, "eps": eps.to_numpy(dtype=float)})
    return timeline.sort_values("effective").reset_index(drop=True)


def average_historical_pe(bundle: dict[str, Any]) -> tuple[float | None, int]:
    prices = bundle.get("prices")
    if prices is None or prices.empty or "Close" not in prices.columns:
        return None, 0

    timeline = build_eps_timeline(bundle.get("income"))
    if timeline is None or timeline.empty:
        return None, 0

    monthly = prices[["Close"]].dropna().copy()
    index = pd.to_datetime(monthly.index)
    if index.tz is not None:
        index = index.tz_localize(None)
    monthly.index = index.astype("datetime64[ns]")
    monthly = monthly.reset_index()
    monthly.columns = ["date", "close"]

    merged = pd.merge_asof(
        monthly.sort_values("date"),
        timeline,
        left_on="date",
        right_on="effective",
        direction="backward",
    )
    merged = merged[merged["eps"] > 0].copy()
    if merged.empty:
        return None, 0

    merged["pe"] = merged["close"] / merged["eps"]
    low, high = CONFIG["PE_OUTLIER_BOUNDS"]
    merged = merged[(merged["pe"] > low) & (merged["pe"] < high)]
    if merged.empty:
        return None, 0

    return float(merged["pe"].mean()), int(len(merged))

In [12]:
def compute_growth_metrics(bundle: dict[str, Any]) -> dict[str, float | None]:
    metrics: dict[str, float | None] = {
        "revenue_cagr_3y": None,
        "net_income_cagr_3y": None,
        "roe_avg_hist": None,
        "fy_count": 0,
    }

    income = bundle.get("income")
    if income is None or income.empty:
        return metrics

    periods = pd.to_datetime(income.columns)
    order = periods.argsort()[::-1]
    periods = periods[order]
    metrics["fy_count"] = int(len(periods))

    if len(periods) < 2:
        return metrics

    span_years = (periods[0] - periods[-1]).days / 365.25

    def line_item(key: str) -> pd.Series | None:
        if key not in income.index:
            return None
        return pd.to_numeric(income.loc[key], errors="coerce").iloc[order]

    for metric, key in (("revenue_cagr_3y", "Total Revenue"), ("net_income_cagr_3y", "Net Income")):
        values = line_item(key)
        if values is None:
            continue
        values = values.dropna()
        if len(values) < 2:
            continue
        metrics[metric] = cagr(float(values.iloc[0]), float(values.iloc[-1]), span_years)

    metrics["roe_avg_hist"] = _average_roe(bundle, line_item("Net Income"))
    return metrics


def _average_roe(bundle: dict[str, Any], net_income: pd.Series | None) -> float | None:
    balance = bundle.get("balance")
    if balance is None or balance.empty or net_income is None:
        return None

    equity = None
    for key in ("Stockholders Equity", "Total Stockholder Equity", "Common Stock Equity"):
        if key in balance.index:
            equity = pd.to_numeric(balance.loc[key], errors="coerce")
            break

    if equity is None:
        return None

    equity.index = pd.to_datetime(equity.index)
    income = net_income.dropna()
    income.index = pd.to_datetime(income.index)

    shared = income.index.intersection(equity.index)
    if len(shared) < CONFIG["MIN_FY_FOR_AVERAGES"]:
        return None

    denominator = equity.loc[shared]
    roe = (income.loc[shared] / denominator.where(denominator > 0)) * 100.0
    roe = roe.replace([float("inf"), float("-inf")], pd.NA).dropna()

    return float(roe.mean()) if not roe.empty else None

In [13]:
def check_historical_pe(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    thresholds = CONFIG["STAGE2_METRICS"]
    records = []

    for _, row in df.iterrows():
        record: dict[str, Any] = {
            "name": row["name"],
            "yahoo_symbol": to_yahoo_symbol(row),
            "avg_pe_hist": None,
            "pe_history_months": 0,
            "revenue_cagr_3y": None,
            "net_income_cagr_3y": None,
            "roe_avg_hist": None,
            "fy_count": 0,
            "yf_status": "ok",
        }

        bundle = download_fundamentals(row)
        if bundle is None:
            record["yf_status"] = "no data"
            records.append(record)
            time.sleep(CONFIG["REQUEST_DELAY_SEC"])
            continue

        record["yahoo_symbol"] = bundle["symbol"]

        try:
            record["avg_pe_hist"], record["pe_history_months"] = average_historical_pe(bundle)
            record.update(compute_growth_metrics(bundle))
        except Exception as exc:
            record["yf_status"] = f"error: {type(exc).__name__}"

        records.append(record)
        time.sleep(CONFIG["REQUEST_DELAY_SEC"])

    enriched = df.merge(pd.DataFrame(records), on="name", how="left")
    enriched["hist_pe_status"] = enriched.apply(_historical_pe_verdict, axis=1)
    enriched["stage2_growth_status"] = enriched.apply(
        lambda row: _growth_verdict(row, thresholds), axis=1
    )
    return enriched


def _historical_pe_verdict(row: pd.Series) -> str:
    if not CONFIG["APPLY_HISTORICAL_PE_FILTER"]:
        return "skipped"
    if pd.isna(row.get("avg_pe_hist")):
        return "insufficient history"
    if row.get("pe_history_months", 0) < CONFIG["MIN_PE_HISTORY_MONTHS"]:
        return "insufficient history"
    if pd.isna(row.get("pe")):
        return "no current PE"
    return "below own avg PE" if row["pe"] < row["avg_pe_hist"] else "above own avg PE"


def _growth_verdict(row: pd.Series, thresholds: dict[str, float | None]) -> str:
    checks = [
        ("MIN_NET_INCOME_CAGR_3Y", "net_income_cagr_3y", "Profit CAGR 3Y"),
        ("MIN_REVENUE_CAGR_3Y", "revenue_cagr_3y", "Sales CAGR 3Y"),
        ("MIN_ROE_5Y_AVG", "roe_avg_hist", "Avg ROE"),
    ]

    failures, missing = [], False
    for key, column, label in checks:
        limit = thresholds[key]
        if limit is None:
            continue
        value = row.get(column)
        if value is None or pd.isna(value):
            missing = True
        elif value < limit:
            failures.append(f"{label} {value:.1f} < {limit:g}")

    if failures:
        return ", ".join(failures)
    return "missing data" if missing else "pass"


def filter_stage2_survivors(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    mask = pd.Series(True, index=df.index)

    if CONFIG["APPLY_HISTORICAL_PE_FILTER"]:
        accepted = {"below own avg PE"}
        if CONFIG["KEEP_WHEN_NO_PE_HISTORY"]:
            accepted.add("insufficient history")
        mask &= df["hist_pe_status"].isin(accepted)

    if any(v is not None for v in CONFIG["STAGE2_METRICS"].values()):
        accepted = {"pass", "skipped"}
        if CONFIG["KEEP_WHEN_STAGE2_METRICS_MISSING"]:
            accepted.add("missing data")
        mask &= df["stage2_growth_status"].isin(accepted)

    return df[mask].reset_index(drop=True)

## 4. Execution & Pipeline Trigger

`run_screen()` is the single entry point. It executes both stages and returns
every intermediate artefact, so the funnel can be inspected after the fact
without rerunning the pipeline.

Stage 2b is network bound and issues several requests per surviving company, so
expect this cell to take roughly a minute.

In [14]:
def add_waiver_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    waivers = []

    for _, row in df.iterrows():
        flags = []
        if row.get("industry_pe_status") in {"insufficient peers", "no industry benchmark"}:
            flags.append("no peers")
        if row.get("hist_pe_status") == "insufficient history":
            flags.append("no PE history")
        if row.get("stage2_growth_status") == "missing data":
            flags.append("no fundamentals")
        waivers.append(", ".join(flags))

    df["waiver"] = waivers
    return df


def run_screen(diagnose: bool = True) -> dict[str, pd.DataFrame]:
    artefacts: dict[str, pd.DataFrame] = {}

    if diagnose:
        artefacts["funnel"] = run_funnel_analysis()

    stage1 = fetch_tradingview_data()
    artefacts["stage1"] = stage1
    if stage1.empty:
        artefacts["shortlist"] = stage1
        return artefacts

    stage2a = apply_industry_pe_filters(stage1)
    artefacts["stage2a"] = stage2a
    if stage2a.empty:
        artefacts["shortlist"] = stage2a
        return artefacts

    annotated = check_historical_pe(stage2a)
    artefacts["annotated"] = annotated
    artefacts["shortlist"] = add_waiver_column(filter_stage2_survivors(annotated))
    return artefacts

In [15]:
results = run_screen()

summary = pd.DataFrame(
    [
        {"Stage": "1. Quality & growth screen", "Companies": len(results.get("stage1", []))},
        {"Stage": "2a. Below industry PE", "Companies": len(results.get("stage2a", []))},
        {"Stage": "2b. Below own historical PE", "Companies": len(results.get("shortlist", []))},
    ]
)
summary

,Stage,Companies
0,1. Quality & growth screen,31
1,2a. Below industry PE,20
2,2b. Below own historical PE,5


### Stage 1 funnel

Survivor counts after each cumulative filter. The largest single reduction
identifies the binding constraint on the screen.

In [16]:
results["funnel"]

,Filter,Threshold,Listings,Removed
0,Exchange (NSE / BSE),,7815,0
1,Market Cap,"> 4,000 Cr",1860,5955
2,ROE,> 20,460,1400
3,ROIC,> 20,337,123
4,Sales Growth 1Y,> 20,137,200
5,Sales Growth 5Y,> 16,97,40
6,PEG,< 1.3,58,39


### Stage 2 rejection analysis

Why candidates were eliminated after passing the quality and growth screen.

In [17]:
annotated = results.get("annotated")

if annotated is not None and not annotated.empty:
    rejections = pd.concat(
        [
            annotated["hist_pe_status"].value_counts().rename("Count").rename_axis("Verdict").to_frame().assign(Test="Historical PE"),
            annotated["stage2_growth_status"].value_counts().rename("Count").rename_axis("Verdict").to_frame().assign(Test="Derived growth"),
        ]
    ).reset_index()[["Test", "Verdict", "Count"]]
else:
    rejections = pd.DataFrame(columns=["Test", "Verdict", "Count"])

rejections

,Test,Verdict,Count
0,Historical PE,below own avg PE,14
1,Historical PE,above own avg PE,3
2,Historical PE,insufficient history,3
3,Derived growth,pass,9
4,Derived growth,"Profit CAGR 3Y 19.6 < 20, Sales CAGR 3Y 13.9 < 17",1
5,Derived growth,"Profit CAGR 3Y 9.6 < 20, Sales CAGR 3Y 11.7 < 17",1
6,Derived growth,Avg ROE 13.0 < 20,1
7,Derived growth,Avg ROE 18.6 < 20,1
8,Derived growth,Avg ROE 19.4 < 20,1
9,Derived growth,"Sales CAGR 3Y 16.6 < 17, Avg ROE 15.8 < 20",1


## 5. Results & Final Shortlist

Companies clearing every filter, with the metrics supporting each decision.

Reading the table:

- `PE` against `IndPE` shows the discount to the industry benchmark
- `PE` against `AvgPE` shows the discount to the stock's own trading history
- `PEmo` is the months of reconstructed PE history behind `AvgPE`
- `Waiver` is non empty where a test could not be evaluated, meaning the stock
  was retained rather than passed

In [18]:
DISPLAY_COLUMNS = [
    ("name", "Symbol"),
    ("exchange", "Exch"),
    ("industry", "Industry"),
    ("close", "Price"),
    ("market_cap", "MktCap"),
    ("pe", "PE"),
    ("industry_pe", "IndPE"),
    ("avg_pe_hist", "AvgPE"),
    ("pe_history_months", "PEmo"),
    ("roe", "ROE"),
    (None, "ROCE"),
    ("revenue_growth_ttm", "Sales1Y"),
    ("revenue_cagr_3y", "Sales3Y"),
    ("revenue_cagr_5y", "Sales5Y"),
    ("net_income_cagr_3y", "Profit3Y"),
    ("peg", "PEG"),
    ("growth_status", "Status"),
    ("waiver", "Waiver"),
]


def format_shortlist(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=[header for _, header in DISPLAY_COLUMNS])

    capital_efficiency = roce_field()
    table = pd.DataFrame(index=df.index)

    for source, header in DISPLAY_COLUMNS:
        column = capital_efficiency if header == "ROCE" else source
        if column is None or column not in df.columns:
            continue

        values = df[column]
        if header == "MktCap":
            table[header] = values.map(crores)
        elif pd.api.types.is_numeric_dtype(values):
            table[header] = pd.to_numeric(values, errors="coerce").round(2)
        else:
            table[header] = values

    if "Industry" in table.columns:
        table["Industry"] = table["Industry"].astype(str).str.slice(0, 24)

    return table.sort_values("PEG").reset_index(drop=True)


shortlist = format_shortlist(results["shortlist"])
shortlist

,Symbol,Exch,Industry,Price,MktCap,PE,IndPE,AvgPE,PEmo,ROE,ROCE,Sales1Y,Sales3Y,Sales5Y,Profit3Y,PEG,Status,Waiver
0,HBLENGINE,NSE,Electrical Products,712.90,"18,608 Cr",25.29,49.11,75.20,39,44.09,43.61,62.97,24.89,29.35,69.53,0.18,High Growth,
1,BAJAJ_AUTO,NSE,Motor Vehicles,"11,793.00","327,012 Cr",28.05,30.29,32.13,39,29.03,22.40,36.91,19.49,17.38,21.03,0.51,High Growth,
2,BLS,NSE,Data Processing Services,273.92,"11,171 Cr",16.01,25.83,43.19,39,32.74,28.58,31.81,18.58,44.35,35.99,0.64,High Growth,
3,GRSE,NSE,Trucks/Construction/Farm,"2,617.80","30,227 Cr",37.46,44.84,51.46,39,31.79,31.53,39.64,28.76,43.75,34.56,0.88,High Growth,
4,NETWEB,NSE,Computer Peripherals,"5,419.80","28,824 Cr",118.07,118.07,157.27,37,32.84,32.48,107.70,49.17,72.54,44.71,1.10,High Growth,no peers


### Discount analysis

The margin by which each candidate trades below its two valuation benchmarks.

In [19]:
final = results["shortlist"]

if final.empty:
    discounts = pd.DataFrame(columns=["Symbol", "PE", "Discount to industry %", "Discount to own history %"])
else:
    discounts = pd.DataFrame(
        {
            "Symbol": final["name"],
            "PE": final["pe"].round(2),
            "Discount to industry %": ((1 - final["pe"] / final["industry_pe"]) * 100).round(1),
            "Discount to own history %": ((1 - final["pe"] / final["avg_pe_hist"]) * 100).round(1),
        }
    ).sort_values("Discount to own history %", ascending=False).reset_index(drop=True)

discounts

,Symbol,PE,Discount to industry %,Discount to own history %
0,HBLENGINE,25.29,48.50,66.40
1,BLS,16.01,38.00,62.90
2,GRSE,37.46,16.50,27.20
3,NETWEB,118.07,0.00,24.90
4,BAJAJ_AUTO,28.05,7.40,12.70


### Export

In [20]:
output = add_waiver_column(results["shortlist"])
output.to_csv(CONFIG["OUTPUT_CSV"], index=False)

print(f"{len(output)} companies written to {CONFIG['OUTPUT_CSV']}")

5 companies written to results.csv



### Tuning the screen

If the shortlist is empty, consult the Stage 1 funnel above to identify the
binding filter before adjusting anything. In practice the sales growth and PEG
constraints bind first.

| Parameter | Default | Suggested relaxation |
|---|---|---|
| `MIN_REVENUE_GROWTH_TTM` | 20 | 12 to 15 |
| `MIN_ROE` / `MIN_ROCE` | 20 | 15 |
| `MAX_PEG` | 1.3 | 2.0, or `None` |
| `STAGE2_METRICS` | 20 / 17 / 20 | `None` to skip the derived growth tests |
| `APPLY_HISTORICAL_PE_FILTER` | `True` | `False` |
| `MIN_MARKET_CAP` | 4,000 Cr | 1,000 Cr to include small caps |

### Limitations

- Screens on reported fundamentals only: no leverage, cash flow quality, or
  governance analysis.
- Reported figures may lag or be restated, and consolidated versus standalone
  treatment varies by company.
- PEG is populated for roughly two thirds of the universe. Companies without it
  are excluded by the `MAX_PEG` filter rather than passed silently.
- Not investment advice.